# Classification Basics


**Before running this notebook, make sure to check `00_environment_setup.md` to see whether you have set the environment correctly.**

In this notebook we'll provide a basic guide to classification using the sklearn library. Make sure to check the `01_classification_metrics.ipynb` notebook for an introduction to classification metrics.



## Outline

1. Recap: What is the classification process?
2. Dataset Description
3. Data splits
4. Binary classification
5. Multi-label classification
6. Multi-class classification

## 1. Recap: What is the classification process?

- We select features to describe our data 
    * We provide a number of features for you, derived from standard signal processing techniques for audio classification.
- We label each datapoint based on what we want the model to learn.
    * You did that during the data annotation phase!
    * Ideally, during the data exploration phase, you gained some insights into which features might be correlated with each other, and which features are more correlated with the labels, and thus could help us predict the labels.
- The features + labels creates our **dataset**
- We pass this to a model and hope it finds some connections between the features and the labels training.

In [1]:
import itertools
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import glob
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from typing import Tuple, List, Dict, Optional, Literal, Union

## 2. Dataset Description

The dataset was **recorded and annotated by you** during the first phases of the project.

### Folder structure

```
MLPC2026_dataset_development.zip
|-- metadata.csv
|-- annotations.csv
\-- audio_features/
    |-- 000001.npz
    |-- 000002.npz
    \-- ...
```

- ```metadata.csv``` contains metadata information about each file.
- ```annotations.csv``` contains the label annotations and their regions (onsets + offsets).
- ```audio_features/``` contains the precomputed audio features.

We also provide an additional zip archive containing the raw audio files, should you wish to use them (not covered in this tutorial). Find more details about the features and metadata in the [Data Exploration Task description](https://moodle.jku.at/pluginfile.php/13827494/mod_resource/content/8/MLPC_2026S___Data_Exploration-1.pdf).


In [2]:
# TODO: MAKE SURE TO CHANGE THE PATH TO THE DATASET HERE!
# Remember that Windows and Linux/Mac handle paths differently.
PATH_TO_DATASET = os.path.join("..", "data", "MLPC2026_dataset_development")

assert os.path.exists(
    PATH_TO_DATASET
), "The dataset folder 'MLPC2026_dataset' does not exist; download the data set and extract its content."
assert os.path.exists(
    os.path.join(PATH_TO_DATASET, "annotations.csv")
), "The file 'MLPC2026_dataset/annotations.csv' does not exist."
assert os.path.exists(
    os.path.join(PATH_TO_DATASET, "metadata.csv")
), "The file 'MLPC2026_dataset/metadata.csv' does not exist."
assert os.path.exists(
    os.path.join(PATH_TO_DATASET, "audio_features")
), "The folder 'MLPC2026_dataset/audio_features' does not exist."

In [3]:
rng = np.random.default_rng(seed=42)

all_audio_features_paths = glob.glob(
    os.path.join(PATH_TO_DATASET, "audio_features", "*.npz")
)
# TODO: Instead of using all files, we will choose just a few,
# otherwise it will take a long time to do all of the experiments
select_subset = 0.2
print(f"Selecting {select_subset*100}% of the dataset")
audio_features_paths = rng.choice(
    all_audio_features_paths,
    size=int(len(all_audio_features_paths) * select_subset),
    replace=False,
).tolist()

Selecting 20.0% of the dataset


As a reminder, here is the information contained in each of the files:

In [4]:
# These are the features computed
# using signal processing techniques
# directly on the audio files.
FEATURE_NAMES = [
    "zcr_mean",
    "zcr_std",
    "zcr_min",
    "zcr_max",
    "melspect_mean",
    "melspect_std",
    "melspect_min",
    "melspect_max",
    "mfcc_mean",
    "mfcc_std",
    "mfcc_min",
    "mfcc_max",
    "mfcc_d_mean",
    "mfcc_d_std",
    "mfcc_d_min",
    "mfcc_d_max",
    "mfcc_d2_mean",
    "mfcc_d2_std",
    "mfcc_d2_min",
    "mfcc_d2_max",
    "flux_mean",
    "flux_std",
    "flux_min",
    "flux_max",
    "flatness_mean",
    "flatness_std",
    "flatness_min",
    "flatness_max",
    "centroid_mean",
    "centroid_std",
    "centroid_min",
    "centroid_max",
    "bandwidth_mean",
    "bandwidth_std",
    "bandwidth_min",
    "bandwidth_max",
    "contrast_mean",
    "contrast_std",
    "contrast_min",
    "contrast_max",
    "rolloff_low_mean",
    "rolloff_low_std",
    "rolloff_low_min",
    "rolloff_low_max",
    "rolloff_high_mean",
    "rolloff_high_std",
    "rolloff_high_min",
    "rolloff_high_max",
    "energy_mean",
    "energy_std",
    "energy_min",
    "energy_max",
    "power_mean",
    "power_std",
    "power_min",
    "power_max",
]

# These are meta data contained in each of the files
METADATA_NAMES = [
    "start_time",
    "end_time",
    "annotations",
    "is_own_recording",
    "class_names",
    "annotator_ids",
    "target_classes",
    "non_target_classes",
    "recording_device",
    "recording_environments",
    "scene_description",
    "device_placement",
]

# Finally, as a reminder, these are the names of our target classes
# which you helped annotate. Thank you everyone for your effort! ;)
CLASSES_NAMES = [
    "bell_ringing",
    "coffee_machine",
    "cutlery_dishes",
    "door_open_close",
    "footsteps",
    "keyboard_typing",
    "keychain",
    "light_switch",
    "microwave",
    "phone_ringing",
    "running_water",
    "toilet_flushing",
    "vacuum_cleaner",
    "wardrobe_drawer_open_close",
    "window_open_close",
]

We then specify a function to load the data. Note that this function supports three modes: "binary", "multiclass" and "multilabel" classification (we learned about them in our previous notebook `01_classification_metrics.ipynb`, so check it out if you need a reminder.)

In [5]:
def get_features_and_targets(
    audio_features: np.ndarray,
    label_type: Literal["binary", "multiclass", "multilabel"] = "multilabel",
) -> Tuple[np.ndarray, Union[np.ndarray, Dict[str, np.ndarray]]]:
    """Load audio features and aggregated target labels from a .npz file.

    Parameters
    ----------
    audio_features : np.ndarray
        .npz file containing audio features and annotations.
    label_type : {"binary", "multiclass", "multilabel"}
        How to encode the targets:
        - "multilabel": binary matrix of shape (T, C), one column per class.
        - "multiclass": integer class index of shape (T, 1), argmax over classes.
        - "binary": dict mapping each class name to a binary array of shape (T, 1).

    Returns
    -------
    x : np.ndarray of shape (T, D)
        Feature matrix with T time frames and D feature dimensions.
    y : np.ndarray of shape (T, C) or (T, 1), or Dict[str, np.ndarray]
        Labels in the format specified by ``label_type``.
    """
    annotations = audio_features["annotations"]
    T, C, _ = annotations.shape

    # Concatenate all features
    x = np.empty(
        (
            T,
            sum(
                audio_features[f].shape[1] if audio_features[f].ndim > 1 else 1
                for f in FEATURE_NAMES
            ),
        )
    )
    col = 0
    for feat_name in FEATURE_NAMES:
        feat = audio_features[feat_name]
        width = feat.shape[1] if feat.ndim > 1 else 1
        x[:, col : col + width] = feat if feat.ndim > 1 else feat[:, None]
        col += width

    # Weighted majority vote — recording creator gets 1.2x weight
    is_own = audio_features["is_own_recording"].astype(bool)   # (A,)
    weights = np.where(is_own, 1.2, 1.0)                       # (A,)
    weighted_votes = np.tensordot(annotations, weights, axes=(2, 0)) / weights.sum()  # (T, C)

    if label_type == "multilabel":
        y = (weighted_votes > 0.5).astype(int)
    elif label_type == "multiclass":
        y = np.argmax(weighted_votes, axis=1).astype(int).reshape(T, 1)
    else:  # "binary"
        y = make_binary_targets((weighted_votes > 0.5).astype(int))

    return x, y


def make_binary_targets(Y: np.ndarray) -> Dict[str, np.ndarray]:
    """Convert a multilabel indicator matrix into per-class binary arrays.

    Parameters
    ----------
    Y : np.ndarray of shape (N, C)
        Binary multilabel matrix with N frames and C classes.

    Returns
    -------
    Y_binary : dict of {str: np.ndarray}
        Dictionary mapping each class name in ``CLASSES_NAMES`` to a binary
        array of shape (N, 1).
    """
    return {
        class_name: Y[:, idx].reshape(-1, 1).astype(int)
        for idx, class_name in enumerate(CLASSES_NAMES)
    }

In [6]:
def read_files(
    file_list: List[str],
    label_type: Literal["binary", "multiclass", "multilabel"] = "multilabel",
) -> Tuple[np.ndarray, Union[np.ndarray, Dict[str, np.ndarray]]]:
    """Load and stack features and labels from a list of .npz files.

    Parameters
    ----------
    file_list : list of str
        Paths to .npz audio feature files to load.
    label_type : {"binary", "multiclass", "multilabel"}
        Passed through to ``get_features_and_targets``; controls the format of
        the returned labels.

    Returns
    -------
    X : np.ndarray of shape (N, D)
        Stacked feature matrix across all files.
    Y : np.ndarray of shape (N, C) or (N, 1), or Dict[str, np.ndarray]
        Stacked labels in the format specified by ``label_type``.
    creator_id : list of creator_ids for each sample
    """
    if label_type not in ("binary", "multiclass", "multilabel"):
        raise ValueError(
            f"`label_type` must be 'binary', 'multiclass', or 'multilabel', got {label_type!r}"
        )
    X = []
    Y = []
    creator_id = []

    for filename in file_list:
        audio_features = dict(np.load(filename, allow_pickle=True))
        
        x, y = get_features_and_targets(audio_features, label_type=label_type)
        X.append(x)
        Y.append(y)

        annotator_ids = audio_features["annotator_ids"]
        mask = audio_features["is_own_recording"]
        creator = annotator_ids[mask]
        creator_id.append(creator)

    if label_type == "binary":
        return X, {cls: [y[cls] for y in Y] for cls in CLASSES_NAMES}, creator_id
    else:
        return X, np.vstack(Y), creator_id

### Task 0: Baseline Classifier

In [7]:
# solution
class BaselineClassifier(BaseEstimator, ClassifierMixin):

    def __init__(self):
        self.majority_class = None

    def fit(self, x_train: np.ndarray, y_train: np.ndarray) -> None:
        """Fit the classifier by finding the majority class.

        Parameters
        ----------
        x_train : np.ndarray of shape (N, D)
            Feature matrix with N samples and D dimensions. Not used directly,
            included for API consistency.
        y_train : np.ndarray of shape (N,)
            Binary labels (0 or 1) for each training sample.
        """
        self.classes_ = np.unique(y_train)
        self.majority_class = 1 if sum(y_train) > len(y_train) / 2 else 0

    def predict(self, x: np.ndarray) -> np.ndarray:
        """Predict binary labels for each sample.

        Parameters
        ----------
        x : np.ndarray of shape (N, D)
            Feature matrix with N samples and D dimensions.

        Returns
        -------
        predictions : np.ndarray of shape (N,)
            Predicted binary labels (all equal to the majority class).
        """
        predictions = np.zeros(x.shape[0]) + self.majority_class
        return predictions

## 3. Data Split

### Splits per class

In [8]:
# Idea is to split the dataset differently depending on the class, to keep class distribution equal while avoiding data leakage through having
# different timesteps from the same sample in both train(cross val) and test. We also want to avoid dataset bias through using the same creator in either set never in both

# first get all the data. start with less from the data and then go more once the thing is running
all_audio_features_paths = glob.glob(
    os.path.join(PATH_TO_DATASET, "audio_features", "*.npz")
)

select_subset = 1

audio_features_paths = rng.choice(
    all_audio_features_paths,
    size=int(len(all_audio_features_paths) * select_subset),
    replace=False,
).tolist()
print(len(audio_features_paths))

3656


In [9]:
X, Y, creator_ids = read_files(audio_features_paths, label_type="binary")

label_dict = {}
for cls in CLASSES_NAMES:
    cls_y_train = Y[cls]

    label_list = []
    for i in range(len(cls_y_train)):
        label = int(cls_y_train[i].any())
        label_list.append(label)

    label_dict[cls] = label_list
    
    print(f"Class {cls} is existent in {np.sum(label_list) / len(label_list):.2%} of the dataset")

Class bell_ringing is existent in 7.36% of the dataset
Class coffee_machine is existent in 5.55% of the dataset
Class cutlery_dishes is existent in 20.98% of the dataset
Class door_open_close is existent in 29.98% of the dataset
Class footsteps is existent in 47.16% of the dataset
Class keyboard_typing is existent in 19.61% of the dataset
Class keychain is existent in 18.24% of the dataset
Class light_switch is existent in 7.47% of the dataset
Class microwave is existent in 10.50% of the dataset
Class phone_ringing is existent in 20.62% of the dataset
Class running_water is existent in 26.86% of the dataset
Class toilet_flushing is existent in 10.97% of the dataset
Class vacuum_cleaner is existent in 8.78% of the dataset
Class wardrobe_drawer_open_close is existent in 14.82% of the dataset
Class window_open_close is existent in 12.50% of the dataset


In [10]:
# I need to fix  creator ids. it has NaNs. We introduce the Nan creator id -1 which is its own group
def clean_creator(creator_ids: list):
    clean_creator_ids = []

    for c in creator_ids:
        c_arr = np.asarray(c)

        if c_arr.size == 0:
            val = -1
        else:
            val = c_arr.item()

        clean_creator_ids.append(val)
    return clean_creator_ids

In [11]:
# SECOND APPROACH: only group recordings by creators, no stratification. Check class distributions after the split manually and adjust seed if they are skewed too much.
clean_creator_ids = clean_creator(creator_ids)

unique_creators = list(set(clean_creator_ids))
train_creators, test_creators = train_test_split(
    unique_creators,
    test_size=0.2,
    random_state=6, # empirically found to give a good class distribution in train and test set
)

# for each file, check which split the creator belongs to
train_file_indices = []
test_file_indices = []

for file_idx, creator in enumerate(clean_creator_ids):
    if creator in train_creators:
        train_file_indices.append(file_idx)
    else:
        test_file_indices.append(file_idx)

train_files_creator = [audio_features_paths[i] for i in train_file_indices]
test_files_creator  = [audio_features_paths[i] for i in test_file_indices]

print(f"Creators  — train: {len(train_creators)}, test: {len(test_creators)}")
print(f"Recordings — train: {len(train_files_creator)}, test: {len(test_files_creator)}")

# verify class distribution
print(f"\n{'Class':<30} {'Overall':>8} {'Train':>8} {'Test':>8}")
print("-" * 58)
for cls in CLASSES_NAMES:
    overall = np.mean(label_dict[cls])
    train   = np.mean([label_dict[cls][i] for i in train_file_indices])
    test    = np.mean([label_dict[cls][i] for i in test_file_indices])
    print(f"{cls:<30} {overall:>8.2%} {train:>8.2%} {test:>8.2%}")

Creators  — train: 286, test: 72
Recordings — train: 2586, test: 1070

Class                           Overall    Train     Test
----------------------------------------------------------
bell_ringing                      7.36%    7.27%    7.57%
coffee_machine                    5.55%    5.80%    4.95%
cutlery_dishes                   20.98%   21.62%   19.44%
door_open_close                  29.98%   29.74%   30.56%
footsteps                        47.16%   47.14%   47.20%
keyboard_typing                  19.61%   19.45%   20.00%
keychain                         18.24%   18.64%   17.29%
light_switch                      7.47%    7.12%    8.32%
microwave                        10.50%   10.83%    9.72%
phone_ringing                    20.62%   20.65%   20.56%
running_water                    26.86%   26.68%   27.29%
toilet_flushing                  10.97%   11.14%   10.56%
vacuum_cleaner                    8.78%    8.39%    9.72%
wardrobe_drawer_open_close       14.82%   14.97%   14.49%


### Preprocessing


In [12]:
# Feature Selection, Normalisation, Feature Engineering
# Feature Selection. Choose the same as for t-SNE in previous task. Taken by visually analysing the correlation matrix. I would suggest to trying some of this
feature_select = [
    "zcr_mean", "zcr_std", "zcr_min", "zcr_max",
    "mfcc_mean", "mfcc_std", "mfcc_min", "mfcc_max",
    "mfcc_d_mean", "mfcc_d_std", "mfcc_d_min", "mfcc_d_max",
    "mfcc_d2_mean", "mfcc_d2_std", "mfcc_d2_min", "mfcc_d2_max",
    "contrast_mean", "contrast_std", "contrast_min", "contrast_max",
    "power_mean", "power_std", "power_min", "power_max"
]
all_feature_names = FEATURE_NAMES + METADATA_NAMES
# Let ChatGPT make a mapping to find the important features in the stacked form
FEATURE_MAP = {
    # ZCR (1 dim × 4 stats)
    "zcr_mean": slice(0, 1),
    "zcr_std": slice(1, 2),
    "zcr_min": slice(2, 3),
    "zcr_max": slice(3, 4),

    # melspect (128 × 4 = 512)
    "melspect_mean": slice(4, 132),
    "melspect_std": slice(132, 260),
    "melspect_min": slice(260, 388),
    "melspect_max": slice(388, 516),

    # mfcc (32 × 4 = 128)
    "mfcc_mean": slice(516, 548),
    "mfcc_std": slice(548, 580),
    "mfcc_min": slice(580, 612),
    "mfcc_max": slice(612, 644),

    # mfcc_d
    "mfcc_d_mean": slice(644, 676),
    "mfcc_d_std": slice(676, 708),
    "mfcc_d_min": slice(708, 740),
    "mfcc_d_max": slice(740, 772),

    # mfcc_d2
    "mfcc_d2_mean": slice(772, 804),
    "mfcc_d2_std": slice(804, 836),
    "mfcc_d2_min": slice(836, 868),
    "mfcc_d2_max": slice(868, 900),

    # flux (1 × 4)
    "flux_mean": slice(900, 901),
    "flux_std": slice(901, 902),
    "flux_min": slice(902, 903),
    "flux_max": slice(903, 904),

    # flatness
    "flatness_mean": slice(904, 905),
    "flatness_std": slice(905, 906),
    "flatness_min": slice(906, 907),
    "flatness_max": slice(907, 908),

    # centroid
    "centroid_mean": slice(908, 909),
    "centroid_std": slice(909, 910),
    "centroid_min": slice(910, 911),
    "centroid_max": slice(911, 912),

    # bandwidth
    "bandwidth_mean": slice(912, 913),
    "bandwidth_std": slice(913, 914),
    "bandwidth_min": slice(914, 915),
    "bandwidth_max": slice(915, 916),

    # contrast (7 × 4 = 28)
    "contrast_mean": slice(916, 923),
    "contrast_std": slice(923, 930),
    "contrast_min": slice(930, 937),
    "contrast_max": slice(937, 944),

    # rolloff_low
    "rolloff_low_mean": slice(944, 945),
    "rolloff_low_std": slice(945, 946),
    "rolloff_low_min": slice(946, 947),
    "rolloff_low_max": slice(947, 948),

    # rolloff_high
    "rolloff_high_mean": slice(948, 949),
    "rolloff_high_std": slice(949, 950),
    "rolloff_high_min": slice(950, 951),
    "rolloff_high_max": slice(951, 952),

    # energy
    "energy_mean": slice(952, 953),
    "energy_std": slice(953, 954),
    "energy_min": slice(954, 955),
    "energy_max": slice(955, 956),

    # power
    "power_mean": slice(956, 957),
    "power_std": slice(957, 958),
    "power_min": slice(958, 959),
    "power_max": slice(959, 960),
}

In [13]:
X_train_pre, Y_train_pre, creator_ids_train = read_files(train_files_creator, label_type="binary")
X_test_pre, Y_test_pre, _ = read_files(test_files_creator, label_type="binary")

In [14]:
# Split train set into folds.
clean_creator_train = clean_creator(creator_ids_train)
groups = np.array(clean_creator_train)

from sklearn.model_selection import GroupKFold

group_kfold = GroupKFold(n_splits=5, shuffle=True, random_state=6)

In [15]:
# Train_X in 5 folds. Use any class for Y here. its ignored anyway in groupkfold
from sklearn.preprocessing import StandardScaler

X_train = {}
X_val = {}
Y_train = {}
Y_val = {}

for fold, (train_idx, val_idx) in enumerate(group_kfold.split(X_train_pre, Y_train_pre["footsteps"], groups)):

    # X first. subset by indices
    X_process_train = [X_train_pre[i] for i in train_idx]
    X_process_val = [X_train_pre[i] for i in val_idx]
    
    # stack
    X_process_train = np.vstack(X_process_train)
    X_process_val = np.vstack(X_process_val)

    # feature select
    X_selected_train = []
    X_selected_val = []
    for feature in feature_select:
        slc = FEATURE_MAP[feature]
        X_selected_train.append(X_process_train[:, slc])
        X_selected_val.append(X_process_val[:, slc])

    # stack em back
    X_selected_train = np.hstack(X_selected_train)
    X_selected_val = np.hstack(X_selected_val)

    # Normalise
    scaler = StandardScaler()
    X_selected_train = scaler.fit_transform(X_selected_train)
    X_selected_val = scaler.transform(X_selected_val)

    # At this point one can do feature engineering with appropriate heuristic and or expert knowledge

    # Put into dict
    X_train[f"split {fold}"] = X_selected_train
    X_val[f"split {fold}"] = X_selected_val

    # Now Y
    Y_train[f"split {fold}"] = {}
    Y_val[f"split {fold}"] = {}

    # Stack frame-level labels per class for this fold (Y_train_pre[cls] is list of per-file timestep arrays -> subset by file indices first, then vstack to frames)
    for cls in CLASSES_NAMES:
        Y_train[f"split {fold}"][cls] = np.vstack([Y_train_pre[cls][i] for i in train_idx])
        Y_val[f"split {fold}"][cls] = np.vstack([Y_train_pre[cls][i] for i in val_idx])

# Same with test. Make a full train set too to fit a scaler.
X_stacked__full_train = np.vstack(X_train_pre)
X_stacked_test = np.vstack(X_test_pre)

X_selected_full_train = []
X_selected_test = []
for feature in feature_select:
    slc = FEATURE_MAP[feature]
    X_selected_full_train.append(X_stacked__full_train[:, slc])
    X_selected_test.append(X_stacked_test[:, slc])

X_train_full = np.hstack(X_selected_full_train)
X_test = np.hstack(X_selected_test)

# Normalise all of the data.
scaler = StandardScaler()

X_train_full = scaler.fit_transform(X_train_full)
X_test = scaler.transform(X_test)
Y_train_full ={}
Y_test = {}
for cls in CLASSES_NAMES:
    Y_train_full[cls] = np.vstack([Y_train_pre[cls][i] for i in range(len(Y_train_pre[cls]))])
    Y_test[cls] = np.vstack([Y_test_pre[cls][i] for i in range(len(Y_test_pre[cls]))])

In [16]:
print(f"\n{'Fold':<6} {'Class':<30} {'Train':>8} {'Val':>8}")
print("-" * 70)

for fold in X_train.keys():
    diffs = []
    for cls in CLASSES_NAMES:
        y_train = Y_train[fold][cls]
        y_val   = Y_val[fold][cls]

        train   = y_train.mean()
        val     = y_val.mean()

        diffs.append(train - val)

        print(f"{fold:<6} {cls:<30} {train:>8.2%} {val:>8.2%}")
    print("\n")

    diffs = np.array(diffs)
    std_diff = np.std(diffs)
    
    print(f"{fold:<6} std: {std_diff:>15.4f}")
    print("\n")


Fold   Class                             Train      Val
----------------------------------------------------------------------
split 0 bell_ringing                      1.36%    1.59%
split 0 coffee_machine                    4.14%    4.35%
split 0 cutlery_dishes                    6.65%    7.24%
split 0 door_open_close                   4.01%    4.23%
split 0 footsteps                        13.36%   13.52%
split 0 keyboard_typing                   9.56%    9.98%
split 0 keychain                          5.27%    5.16%
split 0 light_switch                      0.41%    0.40%
split 0 microwave                         8.04%    7.55%
split 0 phone_ringing                     6.70%    7.00%
split 0 running_water                    13.41%   12.47%
split 0 toilet_flushing                   3.40%    3.35%
split 0 vacuum_cleaner                    6.09%    7.38%
split 0 wardrobe_drawer_open_close        2.19%    2.28%
split 0 window_open_close                 1.48%    1.29%


split 0 std:   

## Performance Metrics

| Metric                   | Description                                                 | When It's Useful                                                       |
| ------------------------ | ----------------------------------------------------------- | ---------------------------------------------------------------------- |
| **Accuracy**             | % of correct predictions `(TP + TN) / Total`                | Simple, but **misleading with imbalanced data**                        |
| **Precision**            | How many predicted positives were correct: `TP / (TP + FP)` | Important when **false positives** are costly (e.g., spam filters)     |
| **Recall (Sensitivity)** | How many actual positives were caught: `TP / (TP + FN)`     | Critical when **false negatives** are costly (e.g., medical diagnosis) |
| **F1 Score**             | Harmonic mean of precision and recall                       | Good **balance** when classes are imbalanced                           |
| **ROC AUC**              | Measures ranking ability over all thresholds                | Good for **probabilistic models**; not sensitive to threshold          |
| **PR AUC**               | Area under Precision-Recall curve                           | Better than ROC AUC for **imbalanced data**                            |


### Performance on Baseline Classifier

In [17]:
# EVALUATION METRIC: Macro averaged F1 Score, balances precision and recall, good for imbalanced datasets. Ensures that both false positives and false negatives are considered, all classes are treated equally.

# Baseline: per class, always predict majority class
baseline_f1_per_class = {}

for fold in X_train.keys():
    for cls in CLASSES_NAMES:
        clf = BaselineClassifier()
        clf.fit(X_train[fold], Y_train[fold][cls].ravel())
        y_pred = clf.predict(X_val[fold])
        baseline_f1_per_class[cls] = f1_score(Y_val[fold][cls].ravel(), y_pred, average="binary", zero_division=0) # .ravel: flatten to get 1D array of shape (N,)

macro_f1_baseline = np.mean(list(baseline_f1_per_class.values()))

print(f"{'Class':<30} {'F1':>6}")
print("-" * 38)
for cls, score in baseline_f1_per_class.items():
    print(f"{cls:<30} {score:>6.4f}")
print("-" * 38)
print(f"{'Macro-Avg F1':<30} {macro_f1_baseline:>6.4f}")


Class                              F1
--------------------------------------
bell_ringing                   0.0000
coffee_machine                 0.0000
cutlery_dishes                 0.0000
door_open_close                0.0000
footsteps                      0.0000
keyboard_typing                0.0000
keychain                       0.0000
light_switch                   0.0000
microwave                      0.0000
phone_ringing                  0.0000
running_water                  0.0000
toilet_flushing                0.0000
vacuum_cleaner                 0.0000
wardrobe_drawer_open_close     0.0000
window_open_close              0.0000
--------------------------------------
Macro-Avg F1                   0.0000


As expected, the the baseline always predicts "negative" for every frame, so it never produces a true positive for any sound event, making both precision and recall for the positive class zero and therefore F1 = 0.

In [18]:
from collections import defaultdict
import time

def evaluate_cv_binary_classifier(
    classifier_cls,
    X_train,
    Y_train,
    X_val,
    Y_val,
    class_names,
    **clf_kwargs
):
    """
    Evaluate one binary classifier per class over CV folds.

    Parameters
    ----------
    classifier_cls : class
        Example: RandomForestClassifier

    X_train, Y_train, X_val, Y_val : dict
        Dicts indexed by fold.

    class_names : list
        List of class names.

    clf_kwargs : dict
        Parameters passed into classifier constructor.

    Returns
    -------
    per_class_mean_f1 : dict
    macro_f1 : float
    """

    scores_per_class = defaultdict(list)

    for fold in X_train.keys():
        start = time.time()

        for cls in class_names:

            clf = classifier_cls(**clf_kwargs)

            clf.fit(
                X_train[fold],
                Y_train[fold][cls].ravel()
            )

            y_pred = clf.predict(X_val[fold])

            f1 = f1_score(
                Y_val[fold][cls].ravel(),
                y_pred,
                average="binary",
                zero_division=0
            )

            scores_per_class[cls].append(f1)
        
        end = time.time()
        print(f"{fold} took {(end - start):.2f} seconds")

    per_class_mean_f1 = {
        cls: np.mean(scores)
        for cls, scores in scores_per_class.items()
    }

    macro_f1 = np.mean(list(per_class_mean_f1.values()))

    return per_class_mean_f1, macro_f1

## 4. Binary Classification

In [19]:
print("""
======================== DATASET SETUP ========================

Cross-validation splits
-----------------------
X_train[f"split {fold}"]        -> np.ndarray of shape (T, D)
X_val[f"split {fold}"]          -> np.ndarray of shape (T, D)

Y_train[f"split {fold}"][cls]  -> np.ndarray of shape (T, 1)
Y_val[f"split {fold}"][cls]    -> np.ndarray of shape (T, 1)

where:
    T = number of samples
    D = number of features
    cls = one of 15 binary class labels


Full training / test sets
-------------------------
X_train_full                    -> np.ndarray of shape (T, D)
X_test                          -> np.ndarray of shape (T, D)

Y_train_full[cls]               -> np.ndarray of shape (T, 1)
Y_test[cls]                     -> np.ndarray of shape (T, 1)

===============================================================
""")


======================== DATASET SETUP ========================

Cross-validation splits
-----------------------
X_train[f"split {fold}"]        -> np.ndarray of shape (T, D)
X_val[f"split {fold}"]          -> np.ndarray of shape (T, D)

Y_train[f"split {fold}"][cls]  -> np.ndarray of shape (T, 1)
Y_val[f"split {fold}"][cls]    -> np.ndarray of shape (T, 1)

where:
    T = number of samples
    D = number of features
    cls = one of 15 binary class labels


Full training / test sets
-------------------------
X_train_full                    -> np.ndarray of shape (T, D)
X_test                          -> np.ndarray of shape (T, D)

Y_train_full[cls]               -> np.ndarray of shape (T, 1)
Y_test[cls]                     -> np.ndarray of shape (T, 1)




In [20]:
# # Classification party starts now!
# # First Classifier: RandomForest.

# from sklearn.ensemble import RandomForestClassifier

# rf_per_class_f1, rf_macro_f1 = evaluate_cv_binary_classifier(
#     RandomForestClassifier,
#     X_train,
#     Y_train,
#     X_val,
#     Y_val,
#     CLASSES_NAMES,
#     n_estimators=200,
#     n_jobs=-1,
#     random_state=42
# )

# print(f"{'Class':<30} {'F1':>6}")
# print("-" * 38)
# for cls, score in rf_per_class_f1.items():
#     print(f"{cls:<30} {score:>6.4f}")
# print("-" * 38)
# print(f"{'Macro-Avg F1':<30} {rf_macro_f1:>6.4f}")

# Class                              F1
# --------------------------------------
# bell_ringing                   0.3221
# coffee_machine                 0.3113
# cutlery_dishes                 0.2092
# door_open_close                0.0000
# footsteps                      0.0656
# keyboard_typing                0.3752
# keychain                       0.3429
# light_switch                   0.0000
# microwave                      0.3954
# phone_ringing                  0.5678
# running_water                  0.6572
# toilet_flushing                0.1258
# vacuum_cleaner                 0.5775
# wardrobe_drawer_open_close     0.0000
# window_open_close              0.0000
# --------------------------------------
# Macro-Avg F1                   0.2633

In [21]:
# # As an experimental part, here I'll train classifiers only on one of the classes that the RandomForest model had problems with.
# from sklearn.svm import SVC
# from sklearn.naive_bayes import GaussianNB

# per_fold = []

# for fold in X_train.keys():
#     nb = GaussianNB(
#     )
#     nb.fit(
#         X_train[fold], 
#         Y_train[fold]["door_open_close"].ravel()
#     )

#     y_pred = nb.predict(
#         X_val[fold]
#     )

#     f1 = f1_score(
#         Y_val[fold]["door_open_close"].ravel(), 
#         y_pred,
#         average="binary",
#         zero_division=0
#     )

#     per_fold.append(f1)

# print(f"Per-fold F1: {per_fold}")
# print(f"Macro F1: {np.mean(per_fold)}")
# Per-fold F1: [0.14152202937249667, 0.19216182048040456, 0.18218218218218218, 0.13751611516974646, 0.15337107565620176]
# Macro F1: 0.16135064457220633

In [22]:
# nb_per_class_f1, nb_macro_f1 = evaluate_cv_binary_classifier(
#     GaussianNB,
#     X_train,
#     Y_train,
#     X_val,
#     Y_val,
#     CLASSES_NAMES
# )

# print(f"{'Class':<30} {'F1':>6}")
# print("-" * 38)
# for cls, score in nb_per_class_f1.items():
#     print(f"{cls:<30} {score:>6.4f}")
# print("-" * 38)
# print(f"{'Macro-Avg F1':<30} {nb_macro_f1:>6.4f}")

# split 0 took 2.69 seconds
# split 1 took 2.63 seconds
# split 2 took 2.63 seconds
# split 3 took 2.65 seconds
# split 4 took 2.71 seconds
# Class                              F1
# --------------------------------------
# bell_ringing                   0.1439
# coffee_machine                 0.1761
# cutlery_dishes                 0.2574
# door_open_close                0.1614
# footsteps                      0.3433
# keyboard_typing                0.3690
# keychain                       0.2449
# light_switch                   0.0197
# microwave                      0.2372
# phone_ringing                  0.4025
# running_water                  0.4794
# toilet_flushing                0.1251
# vacuum_cleaner                 0.3071
# wardrobe_drawer_open_close     0.0931
# window_open_close              0.0624
# --------------------------------------
# Macro-Avg F1                   0.2282

In [23]:
# from sklearn.tree import DecisionTreeClassifier

# dt_per_class_f1, dt_macro_f1 = evaluate_cv_binary_classifier(
#     DecisionTreeClassifier,
#     X_train,
#     Y_train,
#     X_val,
#     Y_val,
#     CLASSES_NAMES
# )

# print(f"{'Class':<30} {'F1':>6}")
# print("-" * 38)
# for cls, score in dt_per_class_f1.items():
#     print(f"{cls:<30} {score:>6.4f}")
# print("-" * 38)
# print(f"{'Macro-Avg F1':<30} {dt_macro_f1:>6.4f}")

# split 0 took 300.28 seconds
# split 1 took 337.69 seconds
# split 2 took 298.03 seconds
# split 3 took 294.78 seconds
# split 4 took 300.01 seconds
# Class                              F1
# --------------------------------------
# bell_ringing                   0.2658
# coffee_machine                 0.3094
# cutlery_dishes                 0.2615
# door_open_close                0.1491
# footsteps                      0.2896
# keyboard_typing                0.4120
# keychain                       0.3423
# light_switch                   0.0229
# microwave                      0.3700
# phone_ringing                  0.3837
# running_water                  0.5727
# toilet_flushing                0.2214
# vacuum_cleaner                 0.3886
# wardrobe_drawer_open_close     0.0854
# window_open_close              0.0601
# --------------------------------------
# Macro-Avg F1                   0.2756

In [24]:
# from sklearn.neural_network import MLPClassifier
# prec = []
# rec = []
# f1 = []

# for fold in X_train.keys():
#     start = time.time()
#     mlp = MLPClassifier(
#         hidden_layer_sizes=(64, 64, 64, 64, 32),
#         max_iter=200,
#         random_state=42
#     )
#     mlp.fit(
#         X_train[fold],
#         Y_train[fold]["light_switch"].ravel()
#     )
#     y_pred = mlp.predict(
#         X_val[fold]
#     )
#     prec_fold = precision_score(
#         Y_val[fold]["light_switch"].ravel(),
#         y_pred
#     ) 
#     rec_fold = recall_score(
#         Y_val[fold]["light_switch"].ravel(),
#         y_pred
#     )
#     f1_fold = f1_score(
#         Y_val[fold]["light_switch"].ravel(),
#         y_pred        
#     )
#     prec.append(prec_fold)
#     rec.append(rec_fold)
#     f1.append(f1_fold)
#     end = time.time()
#     print(f"{fold} took {(end-start):.2f} seconds")

# print("Class : light_switch")
# print(f"Precision: {np.mean(prec)}")
# print(f"Recall: {np.mean(rec)}")
# print(f"F1: {np.mean(f1)}")

In [25]:
# import torch
# import torch.nn as nn
# import torch.optim as optim

# class ANN(nn.Module):

#     def __init__(self, input_dim):
#         super().__init__()

#         self.network = nn.Sequential(
#             nn.Linear(input_dim, 64),
#             nn.ReLU(),

#             nn.Linear(64, 32),
#             nn.ReLU(),

#             nn.Linear(32, 1)
#         )

#     def forward(self, x):
#         return self.network(x)

In [26]:
# prec = []
# rec = []
# f1 = []


# for fold in X_train.keys():
#     model = ANN(
#         input_dim=420
#     )

#     criterion = nn.BCEWithLogitsLoss()
#     optimiser = optim.Adam(model.parameters(), lr=1e-3)

#     # convert once per fold
#     X_tr = torch.tensor(X_train[fold], dtype=torch.float32)
#     y_tr = torch.tensor(Y_train[fold]["light_switch"], dtype=torch.float32)

#     X_v = torch.tensor(X_val[fold], dtype=torch.float32)
#     y_v = torch.tensor(Y_val[fold]["light_switch"], dtype=torch.float32)


#     epochs = 200

#     for epoch in range(epochs):
#         # Training
#         model.train()

#         # forward pass
#         logits = model(X_tr)
#         loss = criterion(logits, y_tr)

#         # backward pass
#         optimiser.zero_grad()
#         loss.backward()
#         optimiser.step()

#         print(f"Epoch {epoch+1}, Loss: {loss.item():.2f}")

#     # Validation
#     model.eval()
#     with torch.no_grad():
#         val_logits = model(X_v)
#         val_probs = torch.sigmoid(val_logits)
#         val_preds = (val_probs > 0.5).float()

#         print("Pred distribution:", val_preds.unique(return_counts=True))
#         print("True distribution:", y_v.unique(return_counts=True))

#         prec_fold = precision_score(
#             y_v.numpy(),
#             val_preds.numpy(),
#             zero_division=0
#         )

#         rec_fold = recall_score(
#             y_v.numpy(),
#             val_preds.numpy(),
#             zero_division=0
#         )

#         f1_fold = f1_score(
#             y_v.numpy(),
#             val_preds.numpy(),
#             zero_division=0
#         )
#         prec.append(prec_fold)
#         rec.append(rec_fold)
#         f1.append(f1_fold)
        
    
# print("Class : light_switch")
# print(f"Precision: {np.mean(prec)}")
# print(f"Recall: {np.mean(rec)}")
# print(f"F1: {np.mean(f1)}")


In [27]:
from xgboost import XGBClassifier



for n_estimator in [100, 200, 300]:
    for learning_rate in [0.01, 0.1]:
        prec = []
        rec = []
        f1 = []
        for fold in X_train.keys():
            y_train = Y_train[fold]["light_switch"].ravel()

            neg = np.sum(y_train == 0)
            pos = np.sum(y_train == 1)

            start = time.time()
            
            xgb = XGBClassifier(
                n_estimators=n_estimator,
                learning_rate=learning_rate,
                max_depth=6, #1-10ish
                min_child_weight=2, # try
                subsample=0.8, #should help to not train every tree on the same data. might not help because of low targetsize
                scale_pos_weight=(neg/pos), # Increase target weights
                objective="binary:logistic",
                eval_metric="aucpr",
                tree_method="hist",
                n_jobs=-1,
                random_state=42
            )

            xgb.fit(
                X_train[fold],
                Y_train[fold]["light_switch"].ravel()
            )

            y_pred = xgb.predict(
                X_val[fold]
            )

            prec_fold = precision_score(
                Y_val[fold]["light_switch"].ravel(),
                y_pred,
                zero_division=0
            )

            rec_fold = recall_score(
                Y_val[fold]["light_switch"].ravel(),
                y_pred,
                zero_division=0
            )

            f1_fold = f1_score(
                Y_val[fold]["light_switch"].ravel(),
                y_pred,
                zero_division=0
            )

            prec.append(prec_fold)
            rec.append(rec_fold)
            f1.append(f1_fold)

            end = time.time()

            print(f"{fold} took {(end-start):.2f} seconds")

        print("Class : light_switch")
        print(f"Precision: {np.mean(prec):.4f}")
        print(f"Recall   : {np.mean(rec):.4f}")
        print(f"F1       : {np.mean(f1):.4f}")

split 0 took 5.30 seconds
split 1 took 5.11 seconds
split 2 took 5.32 seconds
split 3 took 4.80 seconds
split 4 took 4.59 seconds
Class : light_switch
Precision: 0.0431
Recall   : 0.5624
F1       : 0.0795
split 0 took 4.37 seconds
split 1 took 4.87 seconds
split 2 took 4.65 seconds
split 3 took 4.59 seconds
split 4 took 4.79 seconds
Class : light_switch
Precision: 0.1615
Recall   : 0.1877
F1       : 0.1677
split 0 took 9.18 seconds
split 1 took 9.17 seconds
split 2 took 9.29 seconds
split 3 took 9.33 seconds
split 4 took 8.51 seconds
Class : light_switch
Precision: 0.0557
Recall   : 0.5440
F1       : 0.1001
split 0 took 7.96 seconds
split 1 took 8.11 seconds
split 2 took 8.02 seconds
split 3 took 8.05 seconds
split 4 took 8.15 seconds
Class : light_switch
Precision: 0.2574
Recall   : 0.0372
F1       : 0.0636
split 0 took 12.62 seconds
split 1 took 13.12 seconds
split 2 took 13.41 seconds
split 3 took 13.10 seconds
split 4 took 12.58 seconds
Class : light_switch
Precision: 0.0669
Recall

In [ ]:
# pseudo-sophisticated neural net with focal loss and multiple classification heads

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
import time

# torch dataset class for wrapping numpy arrays, so torch DataLoader can batch em up and shuffle em 
class AudioFrameDataset(Dataset):
    def __init__(self, X: np.ndarray, Y_dict: dict, class_names: list):
        self.X = torch.tensor(X, dtype=torch.float32) # convert X from numpy float64 to torch float32
        Y_matrix = np.hstack([Y_dict[cls] for cls in class_names]) # stack all 15 classes (T, 15) for predicting all classes in a single forward pass
        self.Y = torch.tensor(Y_matrix, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

# focal loss to improve learning on rare positive classes (light_switch, bell_ringing, etc)
def focal_loss(logits: torch.Tensor, targets: torch.Tensor, alpha: float = 0.75, gamma: float = 2.0):

    bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')  # element-wise binary cross-entropy
    p_t = torch.exp(-bce)  # recover predicted probability for correct class / shape (B, 15), values in [0, 1]

    focal_weight = (1.0 - p_t) ** gamma  # gamma: down-weight "easy" examples with high p_t
    alpha_weight = alpha * targets + (1.0 - alpha) * (1.0 - targets)  # alpha: give positive class a higher base weight
    loss = alpha_weight * focal_weight * bce

    return loss.mean() # average over batch and classes

class AudioClassifier(nn.Module):
    def __init__(self, input_dim: int = 420, num_classes: int = 15):
        super().__init__()

        # shared backbone feature extractor for all classes
        self.backbone = nn.Sequential(
            # layer 1: 420 -> 256
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            # layer 2: 256 -> 128
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            # layer 3: 128 -> 64, no dropout here (preserve more info for the heads)
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
        )
        self.heads = nn.ModuleList([nn.Linear(64, 1) for _ in range(num_classes)]) # 15 output heads, one per class

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.backbone(x)
        logits = torch.stack([h(z).squeeze(-1) for h in self.heads], dim=1) # h(z) has shape (B, 1) -> squeeze to (B,) -> stack to get (B, 15)
        return logits

In [31]:
use_cuda = torch.backends.mps.is_available() # torch.cuda.is_available() - changed to MPS for Apple Silicon GPU cores
device = torch.device("mps" if use_cuda else "cpu")
print("Device:", device)

Device: mps


In [ ]:
def train_and_evaluate(
    X_train: np.ndarray,
    Y_train_dict: dict,
    X_val: np.ndarray,
    Y_val_dict: dict,
    class_names: list,
    input_dim: int = 420,
    epochs: int = 100,
    batch_size: int = 512,
    lr: float = 1e-3,
    alpha: float = 0.75,
    gamma: float = 2.0,
):
    
    train_ds = AudioFrameDataset(X_train, Y_train_dict, class_names)
    val_ds   = AudioFrameDataset(X_val,  Y_val_dict,  class_names)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0) # shuffle train but not val
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0)

    model = AudioClassifier(input_dim=input_dim, num_classes=len(class_names))
    model.to(device) # use gpu if available
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs) # decay learning rate from lr to 0 during training
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for X_batch, Y_batch in train_loader:
            X_batch = X_batch.to(device)   
            Y_batch = Y_batch.to(device)
            
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = focal_loss(logits, Y_batch, alpha=alpha, gamma=gamma)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()

        if (epoch + 1) % 20 == 0:
            avg_loss = epoch_loss / len(train_loader)
            print(f"  epoch {epoch+1:>3}/{epochs}  loss={avg_loss:.4f}  lr={scheduler.get_last_lr()[0]:.2e}")

    model.eval()
    all_preds   = []
    all_targets = []

    with torch.no_grad():
        for X_batch, Y_batch in val_loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()
            all_preds.append(preds.cpu().numpy()) # collect batch predictions
            all_targets.append(Y_batch.numpy()) # collect batch true labels

    # stack batches back into single matrix
    all_preds   = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)

    per_class_f1 = {}
    for i, cls in enumerate(class_names):
        per_class_f1[cls] = f1_score(all_targets[:, i], all_preds[:, i], average='binary', zero_division=0)
    macro_f1 = np.mean(list(per_class_f1.values()))
    return per_class_f1, macro_f1

In [ ]:
# cross validation loop 

scores_per_class = defaultdict(list)   # accumulate per-class F1 across folds

for fold in X_train.keys():
    print(f"\n=== {fold} ===")
    start = time.time()
    per_class_f1, macro_f1 = train_and_evaluate(
        X_train[fold],
        Y_train[fold],
        X_val[fold],
        Y_val[fold],
        class_names=CLASSES_NAMES,
        input_dim=420,
        epochs=100,
        batch_size=512,
        lr=1e-3,
        alpha=0.75,   # positive class gets 75% of weight
        gamma=2.0,    # standard focal exponent from the original paper
    )

    for cls, score in per_class_f1.items():
        scores_per_class[cls].append(score)   # store F1 for this fold

    print(f"  macro F1 = {macro_f1:.4f}  ({time.time()-start:.1f}s)")

# average F1 across all 5 folds for each class
ann_per_class_f1 = {cls: np.mean(scores) for cls, scores in scores_per_class.items()}
ann_macro_f1     = np.mean(list(ann_per_class_f1.values()))

print(f"\n{'Class':<30} {'F1':>6}")
print("-" * 38)
for cls, score in ann_per_class_f1.items():
    print(f"{cls:<30} {score:>6.4f}")
print("-" * 38)
print(f"{'Macro-Avg F1':<30} {ann_macro_f1:>6.4f}")


=== split 0 ===
  epoch  20/100  loss=0.0126  lr=9.05e-04
  epoch  40/100  loss=0.0115  lr=6.55e-04
  epoch  60/100  loss=0.0099  lr=3.45e-04
  epoch  80/100  loss=0.0080  lr=9.55e-05
  epoch 100/100  loss=0.0070  lr=0.00e+00
  macro F1 = 0.5075  (273.6s)

=== split 1 ===
  epoch  20/100  loss=0.0128  lr=9.05e-04
  epoch  40/100  loss=0.0117  lr=6.55e-04
  epoch  60/100  loss=0.0101  lr=3.45e-04
  epoch  80/100  loss=0.0081  lr=9.55e-05
  epoch 100/100  loss=0.0072  lr=0.00e+00
  macro F1 = 0.5131  (280.8s)

=== split 2 ===
  epoch  20/100  loss=0.0126  lr=9.05e-04
  epoch  40/100  loss=0.0114  lr=6.55e-04
  epoch  60/100  loss=0.0098  lr=3.45e-04
  epoch  80/100  loss=0.0078  lr=9.55e-05
  epoch 100/100  loss=0.0069  lr=0.00e+00
  macro F1 = 0.4962  (278.5s)

=== split 3 ===
  epoch  20/100  loss=0.0136  lr=9.05e-04
  epoch  40/100  loss=0.0134  lr=6.55e-04
  epoch  60/100  loss=0.0110  lr=3.45e-04
  epoch  80/100  loss=0.0089  lr=9.55e-05
  epoch 100/100  loss=0.0079  lr=0.00e+00
  

In [ ]:
#Class                              F1
#--------------------------------------
#bell_ringing                   0.5213
#coffee_machine                 0.5293
#cutlery_dishes                 0.5789
#door_open_close                0.3584
#footsteps                      0.5096
#keyboard_typing                0.7038
#keychain                       0.5934
#light_switch                   0.1354
#microwave                      0.5986
#phone_ringing                  0.6513
#running_water                  0.7610
#toilet_flushing                0.6106
#vacuum_cleaner                 0.6762
#wardrobe_drawer_open_close     0.2343
#window_open_close              0.1995
#--------------------------------------
#Macro-Avg F1                   0.5108

## 5. Multi-Label Classification

### Summary: Classifier Comparison

The table below compares all three classifiers on the validation set for the selected classes (`bell_ringing`, `vacuum_cleaner`). Both metrics are macro-averaged across both classes.

In [ ]:
classwise_bal_macro = { n: m for n, m in zip(selected_classes, list(zip(baseline_bal_accs, dt_bal_accs, rf_bal_accs)))}

results = pd.DataFrame(
    {
        "Classifier": ["Baseline (Majority Class)", "Decision Tree", "Random Forest"],
    } | classwise_bal_macro | {
        "Macro-Avg Accuracy": [baseline_acc_macro, dt_acc_macro, rf_acc_macro],
        "Macro-Avg Balanced Accuracy": [baseline_bal_acc_macro, dt_bal_acc_macro, rf_bal_acc_macro],
    }
).set_index("Classifier")

results.round(4)

NameError: name 'selected_classes' is not defined